## Set Up Spark Session for Delta Lake on MinIO

This cell configures a SparkSession to work with **Delta Lake** using files stored on **MinIO** (S3-compatible storage).  
The setup includes:

- **Delta Lake support** via `delta-spark` package  
- **S3A configuration** to connect to MinIO with access keys  
- **Hadoop AWS** and **AWS SDK bundle** for S3 integration

The `.config(...)` options enable Delta SQL features, register the Delta catalog, and tell Spark how to connect to MinIO using the S3A protocol.



In [1]:
import os
from pyspark.sql import SparkSession

packages = ",".join([
    "io.delta:delta-spark_2.12:3.2.0",       
    "org.apache.hadoop:hadoop-aws:3.3.4",
    "com.amazonaws:aws-java-sdk-bundle:1.12.530",
])

# Create SparkSession with Delta and MinIO support
spark = (
    SparkSession.builder
      .appName("Delta Lakehouse on MinIO")
      .config("spark.jars.packages", packages)
      # Enable Delta Lake SQL support
      .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
      .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
      # Configure S3A (MinIO) access
      .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000")
      .config("spark.hadoop.fs.s3a.path.style.access", "true") # Enable path-style access (important for MinIO)
      .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false") # Disable SSL for local setup
      .config("spark.hadoop.fs.s3a.access.key", os.environ["MINIO_ROOT_USER"])
      .config("spark.hadoop.fs.s3a.secret.key", os.environ["MINIO_ROOT_PASSWORD"])
      .config("spark.hadoop.fs.s3a.aws.credentials.provider", "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider")
      .getOrCreate()
)
print("Spark:", spark.version)

Spark: 3.5.3


## Create Delta Table from Parquet Files

This step creates a new Delta Lake table `people.staff_delta` in the `people` database.

- The table is created using the `delta` format.
- Data is loaded from the `data.parquet` file stored in MinIO's `raw-data` bucket.
- The table is saved to the Delta-compatible path: `s3a://bronze/delta/people/staff`.

In [2]:
spark.sql("CREATE DATABASE IF NOT EXISTS people")

spark.sql("""
CREATE TABLE IF NOT EXISTS people.staff_delta
USING delta
LOCATION 's3a://bronze/delta/people/staff'
AS
SELECT * FROM parquet.`s3a://raw-data/data.parquet`
""")

DataFrame[]

## Verify Row Count in Delta Table

This query checks how many rows were successfully loaded into the `people.staff_delta` table.  
It confirms that data from `data.parquet` was ingested correctly.

In [3]:
spark.sql("SELECT * FROM people.staff_delta").show()

+----------------+--------+---------+--------------------+----------+
|            name| country|residence|                role|      date|
+----------------+--------+---------+--------------------+----------+
| wallace camargo|  brazil| portugal|       data engineer|2026-10-10|
|  halyna halchuk| ukraine| portugal|     tester engineer|2026-10-10|
|   pedro esteves|portugal| portugal|        data analyst|2026-10-10|
|    shalom andre|  angola| portugal|       data engineer|2026-10-10|
|   rafael arruda|  brazil| portugal|       data engineer|2026-10-10|
|  raphael amorim|  brazil|   brazil|      cloud enginner|2026-10-10|
|  italo mesquita|  brazil| portugal|database administ...|2026-10-10|
|      luiz vitor|  brazil|   brazil|           tech lead|2026-10-10|
|gabriel quintela|  brazil|   brazil|       data engineer|2026-10-10|
+----------------+--------+---------+--------------------+----------+



In [4]:
spark.sql("SELECT COUNT(*) FROM people.staff_delta").show()


+--------+
|count(1)|
+--------+
|       9|
+--------+



## Inspect Delta Table Metadata and History

The first query (`DESCRIBE DETAIL`) shows metadata about the `people.staff_delta` table, such as location, format, size, schema, and creation time.

The second query (`DESCRIBE HISTORY`) displays the table's commit history — including operations like `CREATE`, `WRITE`, or `MERGE`, along with timestamps and user information.

In [5]:
spark.sql(""" 
DESCRIBE DETAIL people.staff_delta;
""").show()

spark.sql(""" 
DESCRIBE HISTORY people.staff_delta;
""").show()

+------+--------------------+--------------------+-----------+--------------------+--------------------+-------------------+----------------+-----------------+--------+-----------+----------+----------------+----------------+--------------------+
|format|                  id|                name|description|            location|           createdAt|       lastModified|partitionColumns|clusteringColumns|numFiles|sizeInBytes|properties|minReaderVersion|minWriterVersion|       tableFeatures|
+------+--------------------+--------------------+-----------+--------------------+--------------------+-------------------+----------------+-----------------+--------+-----------+----------+----------------+----------------+--------------------+
| delta|c14d4ef5-c8fd-4f4...|spark_catalog.peo...|       NULL|s3a://bronze/delt...|2026-10-10 09:00:...|2026-10-10 09:00:10|              []|               []|       1|       1975|        {}|               1|               2|[appendOnly, inva...|
+------+----

## Simulate New Batch Ingestion and Track Changes

Since `data.parquet` is a single small file (no second month to load), a new batch is simulated by
re-inserting a deterministic subset of it — every row where `country = 'brazil'` — into the
`people.staff_delta` table.

After the insert, `DESCRIBE HISTORY` is used again to verify that a new commit was recorded.  
This allows tracking how and when the table was updated — a core feature of Delta Lake's transaction log.


In [6]:

spark.sql("""INSERT INTO people.staff_delta
SELECT * FROM parquet.`s3a://raw-data/data.parquet`
WHERE country = 'brazil';
""")

spark.sql("""DESCRIBE HISTORY people.staff_delta;
""").show()

+-------+-------------------+------+--------+--------------------+--------------------+----+--------+---------+-----------+--------------+-------------+--------------------+------------+--------------------+
|version|          timestamp|userId|userName|           operation| operationParameters| job|notebook|clusterId|readVersion|isolationLevel|isBlindAppend|    operationMetrics|userMetadata|          engineInfo|
+-------+-------------------+------+--------+--------------------+--------------------+----+--------+---------+-----------+--------------+-------------+--------------------+------------+--------------------+
|      1|2026-10-10 09:00:17|  NULL|    NULL|               WRITE|{mode -> Append, ...|NULL|    NULL|     NULL|          0|  Serializable|         true|{numFiles -> 1, n...|        NULL|Apache-Spark/3.5....|
|      0|2026-10-10 09:00:10|  NULL|    NULL|CREATE TABLE AS S...|{partitionBy -> [...|NULL|    NULL|     NULL|       NULL|  Serializable|         true|{numFiles -> 1, 

## Query Delta Table at Specific Versions

These queries read the `people.staff_delta` table as it existed at earlier points in time using Delta Lake’s **time travel** feature:

- `VERSION AS OF 0` returns the original state of the table (after the initial load).
- `VERSION AS OF 1` includes the additional rows from the simulated batch insert.

In [7]:
spark.sql(""" 
SELECT COUNT(*) 
FROM delta.`s3a://bronze/delta/people/staff` VERSION AS OF 0;""").show()

spark.sql(""" 
SELECT COUNT(*) 
FROM delta.`s3a://bronze/delta/people/staff` VERSION AS OF 1;""").show()

+--------+
|count(1)|
+--------+
|       9|
+--------+

+--------+
|count(1)|
+--------+
|      15|
+--------+



## Restore Delta Table to an Earlier Version

This command rolls back the `people.staff_delta` table to version 0 — effectively undoing the last insert.
The `DESCRIBE HISTORY` query then shows a new entry for the `RESTORE` operation, confirming that the rollback was recorded in the Delta transaction log.

In [8]:
spark.sql(""" 
RESTORE TABLE people.staff_delta TO VERSION AS OF 0;""")

spark.sql(""" 
DESCRIBE HISTORY people.staff_delta;""").show()

+-------+-------------------+------+--------+--------------------+--------------------+----+--------+---------+-----------+--------------+-------------+--------------------+------------+--------------------+
|version|          timestamp|userId|userName|           operation| operationParameters| job|notebook|clusterId|readVersion|isolationLevel|isBlindAppend|    operationMetrics|userMetadata|          engineInfo|
+-------+-------------------+------+--------+--------------------+--------------------+----+--------+---------+-----------+--------------+-------------+--------------------+------------+--------------------+
|      2|2026-10-10 09:00:22|  NULL|    NULL|             RESTORE|{version -> 0, ti...|NULL|    NULL|     NULL|          1|  Serializable|        false|{numRestoredFiles...|        NULL|Apache-Spark/3.5....|
|      1|2026-10-10 09:00:17|  NULL|    NULL|               WRITE|{mode -> Append, ...|NULL|    NULL|     NULL|          0|  Serializable|         true|{numFiles -> 1, 

## Delta Lake Change Data Feed (CDF) Demo

This sequence shows how to enable and use CDF on `people.staff_delta`:

1. **Enable CDF**  
   Turn on `delta.enableChangeDataFeed = true` so future commits record row-level changes.

2. **Mark the starting version**  
   Read `DESCRIBE HISTORY` and capture the current table version before making changes.

3. **Make controlled changes**  
   - `INSERT` a tiny, deterministic slice (`country = 'angola'`, a single row).  
   - `UPDATE` a subset (set `residence = 'remote'` for `role = 'tech lead'`).  
   - `DELETE` a few rows (`role = 'cloud enginner'`).

4. **Read only the changes**  
   Load the table with:
   - `readChangeFeed = true`  
   - `startingVersion = <captured version>`  
   - `endingVersion = latest`  
   This returns only changed rows between those versions.

5. **Inspect change types**  
   CDF adds metadata columns:
   - `_change_type`: `insert`, `update_preimage`, `update_postimage`, `delete`  
   - `_commit_version`, `_commit_timestamp`  
   Select relevant business columns plus these fields to see exactly what changed.  
   (Optional) `GROUP BY _change_type` to summarize counts.

Notes:
- CDF starts recording **after** it’s enabled; earlier versions won’t have change rows.
- With only 9 source rows, each change touches very few records — that's expected here.


In [9]:
from pyspark.sql.functions import col

# 1) Enable CDF (applies to future commits)
spark.sql("""
ALTER TABLE people.staff_delta
SET TBLPROPERTIES (delta.enableChangeDataFeed = true)
""")

# 2) Capture the current version BEFORE changes
start_version = (spark.sql("DESCRIBE HISTORY people.staff_delta")
                   .selectExpr("max(version) as v").collect()[0][0])

# 3) Make some changes: INSERT + UPDATE + DELETE
# Insert a small, deterministic slice
spark.sql("""
INSERT INTO people.staff_delta
SELECT *
FROM parquet.`s3a://raw-data/data.parquet`
WHERE country = 'angola'
""")

# Update a small subset
spark.sql("""
UPDATE people.staff_delta
SET residence = 'remote'
WHERE role = 'tech lead'
""")

# Delete a tiny subset
spark.sql("""
DELETE FROM people.staff_delta
WHERE role = 'cloud enginner'
""")

# 4) Read just the changes between start_version -> latest
end_version = (spark.sql("DESCRIBE HISTORY people.staff_delta")
                 .selectExpr("max(version) as v").collect()[0][0])

cdf = (spark.read.format("delta")
       .option("readChangeFeed", "true")
       .option("startingVersion", start_version)
       .option("endingVersion", end_version)
       .table("people.staff_delta"))

# 5) Inspect change types
cdf.select("_commit_version", "_commit_timestamp", "_change_type",
           "name", "country", "role", "residence") \
   .show(truncate=False)

+---------------+-----------------------+----------------+--------------+-------+--------------+---------+
|_commit_version|_commit_timestamp      |_change_type    |name          |country|role          |residence|
+---------------+-----------------------+----------------+--------------+-------+--------------+---------+
|5              |2026-10-10 09:00:27.919|update_preimage |luiz vitor    |brazil |tech lead     |brazil   |
|5              |2026-10-10 09:00:27.919|update_postimage|luiz vitor    |brazil |tech lead     |remote   |
|6              |2026-10-10 09:00:29.52 |delete          |raphael amorim|brazil |cloud enginner|brazil   |
|4              |2026-10-10 09:00:25.632|insert          |shalom andre  |angola |data engineer |portugal |
+---------------+-----------------------+----------------+--------------+-------+--------------+---------+



In [10]:
cdf.groupBy("_change_type").count().show()

+----------------+-----+
|    _change_type|count|
+----------------+-----+
| update_preimage|    1|
|update_postimage|    1|
|          delete|    1|
|          insert|    1|
+----------------+-----+



## Add and Populate a New Column in Delta Table

This step adds a new column `role_group` to the `people.staff_delta` table and fills it with values based on `role`:

- `'leadership'` for `tech lead`  
- `'analytics'` for `data analyst`  
- `'engineering'` for everything else (engineers, DBAs, testers)

The final query groups the data by `role_group` to show the distribution of roles.

In [11]:
# Disabling CDF because of Java OOM
spark.sql("""ALTER TABLE people.staff_delta UNSET TBLPROPERTIES (delta.enableChangeDataFeed);""")

spark.sql("""ALTER TABLE people.staff_delta ADD COLUMNS (role_group STRING);""")

spark.sql("""UPDATE people.staff_delta
SET role_group = CASE
  WHEN role = 'tech lead' THEN 'leadership'
  WHEN role = 'data analyst' THEN 'analytics'
  ELSE 'engineering'
END
WHERE role IS NOT NULL;""")

spark.sql("""SELECT role_group, COUNT(*) 
FROM people.staff_delta 
GROUP BY role_group;""").show()

+-----------+--------+
| role_group|count(1)|
+-----------+--------+
| leadership|       1|
|  analytics|       1|
|engineering|       7|
+-----------+--------+



## Evolve Delta Lake Schema On Write (without ALTER TABLE)

In Delta Lake, you don’t have to use `ALTER TABLE` to add a new column — you can also evolve the schema automatically during write, just like with Iceberg.

I had already added a `role_group` column earlier, but Delta didn’t let me drop it until I enabled column mapping mode (`'name'`). That required a protocol upgrade with `minReaderVersion` and `minWriterVersion`.

Once that was done, I calculated a new numeric column `name_length` (the length of each person's `name`) and appended it directly with `.option("mergeSchema", "true")`. Delta updated the schema during write, added a new table version, and left the old files untouched — only new ones have the new column.

In [12]:
from pyspark.sql.functions import col, length as slen


spark.sql("""ALTER TABLE people.staff_delta SET TBLPROPERTIES (
  'delta.columnMapping.mode' = 'name',
  'delta.minReaderVersion' = '2',
  'delta.minWriterVersion' = '5'
);""")

spark.sql("ALTER TABLE people.staff_delta DROP COLUMN role_group")
# Load the source data again
df = spark.read.parquet("s3a://raw-data/data.parquet")

# Compute the new column
df_with_len = df.withColumn("name_length", slen(col("name")))

# Append with schema evolution
(df_with_len.write
    .format("delta")
    .option("mergeSchema", "true")
    .mode("append")
    .save("s3a://bronze/delta/people/staff"))
spark.sql("DESCRIBE TABLE people.staff_delta;").show(30)

+-----------+---------+-------+
|   col_name|data_type|comment|
+-----------+---------+-------+
|       name|   string|   NULL|
|    country|   string|   NULL|
|  residence|   string|   NULL|
|       role|   string|   NULL|
|       date|   string|   NULL|
|name_length|      int|   NULL|
+-----------+---------+-------+

